In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"
HF_TOKEN = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto"
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [ ]:

import os, re, time, math, json
import pandas as pd
import torch
from tqdm import tqdm


INPUT_XLSX = "/content/Dataset_550LLM.xlsx"

OUT_COT_CSV   = "/content/llama2_7b_cot_output.csv"
OUT_PROGRESS  = "/content/llama2_7b_progress.csv"
OUT_COT_XLSX  = "/content/llama2_7b_cot_output.xlsx"

SAVE_EVERY = 10


assert "model" in globals() and "tokenizer" in globals(), " model/tokenizer not found. Load the model first."

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

device = model.device
print(f" Using device: {device}")

df = pd.read_excel(INPUT_XLSX)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

required_cols = ["PromptID", "English", "Hindi"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f" Missing columns in Excel: {missing}. Need: {required_cols}")


def cot_prompt(user_text: str, lang: str) -> str:
    user_text = str(user_text).strip()
    if lang.lower() == "en":
        instruction = (
            "You are a supportive mental-health peer assistant. "
            "Think step-by-step internally to understand the situation, but DO NOT show your reasoning. "
            "Only output the final helpful response in English. "
            "Be empathetic, non-judgmental, and avoid medical claims. "
            "If the user mentions self-harm/suicide, encourage seeking immediate professional help."
        )
    else:
        instruction = (
            "आप एक सहायक मानसिक-स्वास्थ्य peer assistant हैं। "
            "समस्या को समझने के लिए अंदर ही अंदर step-by-step सोचें, लेकिन अपनी reasoning दिखाएँ नहीं। "
            "केवल अंतिम मददगार उत्तर हिंदी में दें। "
            "सहानुभूतिपूर्ण, non-judgmental रहें और medical claims न करें। "
            "अगर self-harm/suicide का ज़िक्र हो तो तुरंत professional help लेने के लिए कहें।"
        )

    return f"<s>[INST] {instruction}\n\nUser: {user_text} [/INST]"


MAX_NEW_TOKENS = 220
TEMPERATURE = 0.7
TOP_P = 0.9

@torch.inference_mode()
def generate_text(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(out[0], skip_special_tokens=True)


    if "User:" in text:

        pass


    return text.strip()

done_ids = set()
if os.path.exists(OUT_PROGRESS):
    try:
        prog = pd.read_csv(OUT_PROGRESS)
        if "PromptID" in prog.columns:
            done_ids = set(prog["PromptID"].astype(str).tolist())
            print(f" Resuming: {len(done_ids)} already done from progress file.")
    except Exception as e:
        print("⚠️ Could not read progress file, starting fresh.", e)

results = []
progress_rows = []

if os.path.exists(OUT_COT_CSV):
    try:
        existing = pd.read_csv(OUT_COT_CSV)
        results = existing.to_dict("records")
        print(f"📄 Loaded existing output: {len(results)} rows.")
    except Exception as e:
        print("⚠️ Could not read existing output CSV, will overwrite.", e)
        results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    pid = str(row["PromptID"])
    if pid in done_ids:
        continue

    en_text = row["English"]
    hi_text = row["Hindi"]

    en_prompt = cot_prompt(en_text, "en")
    hi_prompt = cot_prompt(hi_text, "hi")


    t0 = time.time()
    en_resp = generate_text(en_prompt)
    hi_resp = generate_text(hi_prompt)
    t1 = time.time()

    results.append({
        "PromptID": pid,
        "English": en_text,
        "Hindi": hi_text,
        "EN_CoT_Prompt": en_prompt,
        "EN_CoT_Response": en_resp,
        "HI_CoT_Prompt": hi_prompt,
        "HI_CoT_Response": hi_resp,
        "seconds": round(t1 - t0, 3),
    })

    progress_rows.append({"PromptID": pid, "status": "done"})


    if (len(progress_rows) % SAVE_EVERY) == 0:
        pd.DataFrame(results).to_csv(OUT_COT_CSV, index=False)
        pd.DataFrame(progress_rows).to_csv(OUT_PROGRESS, mode="a", header=not os.path.exists(OUT_PROGRESS), index=False)
        progress_rows = []
        print(f" Saved @ {pid} (total {len(results)})")

pd.DataFrame(results).to_csv(OUT_COT_CSV, index=False)
if progress_rows:
    pd.DataFrame(progress_rows).to_csv(OUT_PROGRESS, mode="a", header=not os.path.exists(OUT_PROGRESS), index=False)

try:
    pd.DataFrame(results).to_excel(OUT_COT_XLSX, index=False, engine="xlsxwriter")
except Exception as e:
    print(" XLSX export failed, CSV is saved. Error:", e)

print(" DONE")
print("CoT CSV  :", OUT_COT_CSV)
print("Progress :", OUT_PROGRESS)
print("CoT XLSX :", OUT_COT_XLSX)

from google.colab import files
for p in [OUT_COT_CSV, OUT_PROGRESS, OUT_COT_XLSX]:
    if os.path.exists(p):
        files.download(p)


✅ Using device: cuda:0
Rows: 554
Columns: ['PromptID', 'English', 'Hindi']


  2%|▏         | 10/554 [02:28<2:15:59, 15.00s/it]

💾 Saved @ PO10 (total 10)


  4%|▎         | 20/554 [04:59<2:14:05, 15.07s/it]

💾 Saved @ PO20 (total 20)


...

⚠️ XLSX export failed, CSV is saved. Error: No module named 'xlsxwriter'
✅ DONE
CoT CSV  : /content/llama2_7b_cot_output.csv
Progress : /content/llama2_7b_progress.csv
CoT XLSX : /content/llama2_7b_cot_output.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>